In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, DatasetDict
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

In [ ]:
raw_dataset = load_dataset('json', data_files='../datasets/article_summary/valid_finnish_articles_200.json')

original_train = raw_dataset['train']

train_valid = original_train.train_test_split(test_size=0.2, seed=123)

dataset = DatasetDict({
    'train': train_valid['train'],
    'validation': train_valid['test']
})

In [ ]:
model_name = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.gradient_checkpointing_enable()

# Preprocessing function
def preprocess_function(examples):
    inputs = ["tiivistä: " + text for text in examples["text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_ids = np.where(pred_ids != -100, pred_ids, tokenizer.pad_token_id)
    labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    # Compute ROUGE
    rouge_scores = rouge.compute(predictions=pred_str, references=labels_str)

    return {
        "rouge1": round(rouge_scores["rouge1"] * 100, 2),
        "rouge2": round(rouge_scores["rouge2"] * 100, 2),
        "rougeL": round(rouge_scores["rougeL"] * 100, 2),
    }

training_args = Seq2SeqTrainingArguments(
    logging_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=30,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    metric_for_best_model="rouge2",
    greater_is_better=True,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

trainer.save_model("./finetuned-mt5-finnish-summarizer")

In [ ]:
from projects.datascrape import scrape_article
from transformers import pipeline

summarizer = pipeline("text2text-generation", model="./finetuned-mt5-finnish-summarizer", tokenizer="google/mt5-base")
new_text = scrape_article("https://yle.fi/a/74-20158531", "yle")
input_text = "Tiivistä: " + new_text
summary = summarizer(input_text, max_length=100, min_length=50, do_sample=False)

print(summary[0])